## Extract data

In [ ]:
import requests

def fetch_kegg_pathway_text(pathway_id: str) -> str:
    url = f"https://rest.kegg.jp/get/{pathway_id}"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.text

def parse_kegg_entry(text: str) -> dict:
    """
    Very basic parser for KEGG “get” text format.
    You’ll likely need to improve it, especially for nested fields.
    """
    lines = text.splitlines()
    data = {}
    current_key = None
    for line in lines:
        if not line:
            continue
        # The KEGG “get” format usually has a keyword in col 0–10, then data in col 12 onward
        # if starts with a word in the first 10 columns, treat as new key.
        key_candidate = line[:10].strip()
        rest = line[10:].strip()
        if key_candidate and rest:
            # new key (e.g. “ENTRY”, “NAME”, “GENE”, “DRUG”, etc.)
            current_key = key_candidate
            data[current_key] = rest
        else:
            # continuation line
            if current_key is not None:
                data[current_key] += "\n" + line.strip()
    return data

{
  "ENTRY": "hsa05223                    Pathway",
  "NAME": "Non-small cell lung cancer - Homo sapiens (human)",
  "DESCRIPTIO": "N Lung cancer is a leading cause of cancer death among men and women in industrialized countries. Non-small-cell lung cancer (NSCLC) accounts for approximately 85% of lung cancer and represents a heterogeneous group of cancers, consisting mainly of squamous cell (SCC), adeno (AC) and large-cell carcinoma. Molecular mechanisms altered in NSCLC include activation of oncogenes, such as K-RAS, EGFR and EML4-ALK, and inactivation of tumorsuppressor genes, such as p53, p16INK4a, RAR-beta, and RASSF1. Point mutations within the K-RAS gene inactivate GTPase activity and the p21-RAS protein continuously transmits growth signals to the nucleus. Mutations or overexpression of EGFR leads to a proliferative advantage. EML4-ALK fusion leads to constitutive ALK activation, which causes cell proliferation, invasion, and inhibition of apoptosis. Inactivating mutation of p5

In [20]:
network = [x[:7] for x in fetch_kegg_pathway_json('hsa05223')["NETWORK"].split("\n")]

In [24]:
elements = [x[:6] for x in fetch_kegg_pathway_json('hsa05223')["ELEMENT"].split("\n")]
queries = [f"{code}+{"+".join(elements)}" for code in network]

In [36]:
relations_network = [fetch_kegg_pathway_json(query) for query in elements]
relations_network

[{'ENTRY': 'N00007                      Network',
  'NAME': 'EML4-ALK fusion kinase to RAS-ERK signaling pathway',
  'DEFINITION': 'EML4-ALK -> RAS -> RAF -> MEK -> ERK -> CCND1',
  'EXPANDED': '(238v1,238v2) -> (3265,3845,4893) -> (369,673,5894) -> (5604,5605) -> (5594,5595) -> 595',
  'CLASS': 'nt06266 Non-small cell lung cancer\nnt06210 ERK signaling (cancer)',
  'TYPE': 'Variant',
  'PATHWAY': 'hsa05223  Non-small cell lung cancer',
  'DISEASE': 'H00014  Non-small cell lung cancer',
  'GENE': '238  ALK; ALK receptor tyrosine kinase\n3265  HRAS; HRas proto-oncogene, GTPase\n3845  KRAS; KRAS proto-oncogene, GTPase\n4893  NRAS; NRAS proto-oncogene, GTPase\n369  ARAF; A-Raf proto-oncogene, serine/threonine kinase\n673  BRAF; B-Raf proto-oncogene, serine/threonine kinase\n5894  RAF1; Raf-1 proto-oncogene, serine/threonine kinase\n5604  MAP2K1; mitogen-activated protein kinase kinase 1\n5605  MAP2K2; mitogen-activated protein kinase kinase 2\n5594  MAPK1; mitogen-activated protein kinase

In [37]:
graph = {
    relation["ENTRY"].split(" ")[0]: {
        "pathway": [rel[5:] for rel in relation.get("PATHWAY", "").split("\n")],
        "relations": relation.get("DEFINITION", "").split("\n")
    }
    for relation in relations_network
}
graph

{'N00007': {'pathway': ['223  Non-small cell lung cancer'],
  'relations': ['EML4-ALK -> RAS -> RAF -> MEK -> ERK -> CCND1']},
 'N00008': {'pathway': ['216  Thyroid cancer',
   '223  Non-small cell lung cancer'],
  'relations': ['RET* -> RAS -> RAF -> MEK -> ERK']},
 'N00012': {'pathway': ['210  Colorectal cancer',
   '212  Pancreatic cancer',
   '226  Gastric cancer',
   '216  Thyroid cancer',
   '221  Acute myeloid leukemia',
   '213  Endometrial cancer',
   '223  Non-small cell lung cancer',
   '218  Melanoma'],
  'relations': ['(KRAS*,NRAS*) -> RAF -> MEK -> ERK -> CCND1']},
 'N00014': {'pathway': ['223  Non-small cell lung cancer'],
  'relations': ['EGFR* -> GRB2 -> SOS -> RAS -> RAF -> MEK -> ERK -> CCND1']},
 'N00022': {'pathway': ['219  Bladder cancer',
   '224  Breast cancer',
   '223  Non-small cell lung cancer'],
  'relations': ['EGF -> (ERBB2*+EGFR) -> GRB2 -> SOS -> RAS -> RAF -> MEK -> ERK']},
 'N00024': {'pathway': ['223  Non-small cell lung cancer'],
  'relations': ['EG

In [41]:
import json
with open("../data/hsa05223_network.json", "w", encoding="utf-8") as f:
    json.dump(graph, f, ensure_ascii=False, indent=2)